# Multiplexed state preparation by CORDIC arithmetic for the periodic Poisson isometry

The three listings of Section 4 of the paper, with the output printed there. They go from the circuit used as a library, through one wavenumber in its registers, to its cost against the two standard constructions.

Install the package first, from the repository root: `pip install -r requirements.txt`.


## Listing 1. The circuit at $N = 32$ and $F = 32$, simulated bit by bit at every wavenumber

The simplest use of the package: the configuration of the abstract. We build the circuit with `poisson_isometry`, simulate its arithmetic bit by bit for every wavenumber $k \ne 0$, and compare the prepared vector with $c(k)/\|c(k)\|_2$. The checks confirm that the work register returns to $|0\rangle$ and that the bits of $k$ are restored, which is what allows the circuit to be used inside a block encoding. The largest error is below the measured bound $6N\,2^{-F}$ of Section 5. Runs in a few seconds.


In [1]:
import numpy as np
from quantumpoissonhomogenization.arithmetic import (
    poisson_isometry, exact_unit)

sv = poisson_isometry(n=5, F=32)       # N = 32, F = 32
amp, vreg, vex, _, _, ck = sv.simulate()   # every k
err = np.linalg.norm(amp - exact_unit(vex), axis=1)
r = sv.tree.rotation_counts()
tof = sum(sv.toffoli().values()) + r["toffoli"]

print(f"wavenumbers      = {len(err)}")
print(f"bits left set    = {ck['work_bits_left_set']}")
print(f"inputs restored  = {ck['inputs_restored']}")
print(f"max |error|      = {err.max():.1e}")
print(f"6 N 2^-F         = {6 * sv.N * 2.0**-sv.F:.1e}")
print(f"Toffoli gates    = {tof:,}")
print(f"ctrl. rotations  = {r['controlled_ry']}")


wavenumbers      = 1023
bits left set    = 0
inputs restored  = True
max |error|      = 9.0e-09
6 N 2^-F         = 4.5e-08
Toffoli gates    = 23,089
ctrl. rotations  = 65


## Listing 2. The example of Sections 2 and 3: one wavenumber, $k = (3, 1)$, at $N = 16$ and $F = 12$

One level down, to the registers. For one wavenumber, we print the integers $Z$ that Part 1 writes into the registers of $a$, $b$, and $\zeta$, each read as $Z/2^F$, and the vector that Part 2 prepares from them. The integers are the bit strings of the example in Section 2, and the prepared vector and its error are those of the example in Section 3. Runs in under a second.


In [2]:
import numpy as np
from quantumpoissonhomogenization.arithmetic import (
    poisson_isometry, exact_unit)

sv = poisson_isometry(n=4, F=12)       # N = 16, F = 12
amp, vreg, vex, _, _, _ = sv.simulate()
k1, k2 = 3, 1
i = k1 * sv.N + k2 - 1       # k = 0 is skipped
Z = np.round(vreg[i, :3] * [1, 1, 3**0.5] * 2**sv.F)
exact = exact_unit(vex)[i]
d = np.linalg.norm(amp[i] - exact)

print(f"integers of a, b, zeta = {Z.astype(int)}")
print(f"prepared = {np.round(amp[i], 5)}")
print(f"exact    = {np.round(exact, 5)}")
print(f"error    = {d:.1e}")


integers of a, b, zeta = [2229  663  442]
prepared = [0.94709 0.28202 0.10835 0.10835]
exact    = [0.94703 0.28192 0.10876 0.10876]
error    = 5.9e-04


## Listing 3. The T counts of the three constructions at $\delta = 1.96 \times 10^{-10}$

The circuit against the two standard constructions: standard state preparation multiplexed over the wavenumber, and a lookup of tabulated angles in a quantum read-only memory (QROM). The T counts are estimates, from the gate lists for the circuit and from published cost formulas for the other two. The lookup needs the fewest T gates up to $N = 2048$, and the circuit the fewest at $N = 4096$, as in Table 3. Runs in about a second.


In [3]:
from quantumpoissonhomogenization.arithmetic import (
    alternatives)

print("    N   F  T: multiplexor   lookup  circuit")
for N in (16, 1024, 2048, 4096):
    o = alternatives.options(N, delta=1.955e-10)
    m, l, a = (o[k]["T"] for k in
               ("multiplexor", "lookup", "arithmetic"))
    F = o["arithmetic"]["F"]
    print(f"{N:5d}  {F}  {m:14.1e} {l:8.1e} {a:8.1e}")


    N   F  T: multiplexor   lookup  circuit
   16  39         9.6e+04  1.3e+03  1.5e+05
 1024  45         5.1e+08  7.7e+04  2.0e+05
 2048  46         2.1e+09  1.5e+05  2.1e+05
 4096  47         8.7e+09  3.1e+05  2.2e+05
